# pycograd — notebook demo

`%load_ext pycograd` loads [pipescript](https://github.com/smacke/pipescript) (if needed) and enables the DSL. Every model below follows one shape:

```python
with params{ ... } as weights:     # build params, inject them as ambient names
    forward = $ |> ...             # write the forward ONCE, as a pipeline
    objective = lambda: forward(X) |> loss($, target)
    for _ in range(steps):
        value, grads = weights.grad(objective)   # bind weights -> Vars, backprop
        weights.step(grads, lr)                  # in-place SGD
```

No `Var` to construct, no parameters threaded by hand.

**Requires** the `pipescript` extra: `pip install pycograd[notebook]`.

In [1]:
%load_ext pycograd

## The primitive: `grad`

Underneath the DSL, `grad` / `value_and_grad` differentiate an ordinary numpy function — the array argument is lifted onto the tape for you.

In [2]:
import numpy as np
from pycograd import grad

def f(x):
    return np.sum(np.sin(x * x))

x = np.array([0.5, 1.0, 1.5])
(gx,) = grad(f)(x)
print("autodiff :", np.round(gx, 4))
print("analytic :", np.round(2 * x * np.cos(x * x), 4))

autodiff : [ 0.9689  1.0806 -1.8845]
analytic : [ 0.9689  1.0806 -1.8845]


## Setup: data and a loss

There is no op library to import — pycograd differentiates raw numpy. Activations are just numpy used as pipe stages: `relu` is `np.maximum(0.0, $)`, and a linear layer is `$ @ w + b`.

The one thing worth naming is the **loss**. Softmax cross-entropy bundles several `np.*` calls, so we write it once — and because a `np.*` call only differentiates when it is a pipe stage *or* sits inside an instrumented (cell-defined) function, packaging it as a helper is exactly what makes its internal `np.exp` / `np.sum` / `np.log` flow gradients.

In [3]:
def softmax_ce(logits, onehot):
    z = logits - np.max(logits, axis=1, keepdims=True)          # stabilize
    logp = z - np.log(np.sum(np.exp(z), axis=1, keepdims=True)) # log-softmax
    return -np.mean(np.sum(onehot * logp, axis=1))

rng = np.random.default_rng(0)
m = 40
centers = np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.5, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]   # one-hot, 3 classes

## Example 1 — a linear softmax classifier

The forward is just `$ @ w + b` (the logits); the loss does the softmax.

In [4]:
with params{
    w = np.zeros((2, 3))
    b = np.zeros(3)
} as weights:
    forward = $ |> $ @ w + b
    objective = lambda: forward(X) |> softmax_ce($, Y)
    first = last = None
    for _ in range(200):
        value, grads = weights.grad(objective)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.0986 -> 0.0040
train accuracy: 1.0


## Example 2 — a 2-layer MLP, as one pipeline

`relu` is inlined as the pipe stage `np.maximum(0.0, $)` — no helper needed.

In [5]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> np.maximum(0.0, $) |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    first = last = None
    for _ in range(300):
        value, grads = weights.grad(objective)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.5142 -> 0.0007
train accuracy: 1.0


## Example 3 — a frozen parameter

`frozen[...]` holds a weight fixed: its gradient comes back `None` and `weights.step` skips it. Here the first-layer bias never moves while everything else trains.

In [6]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = frozen[np.zeros(16)]
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> np.maximum(0.0, $) |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    for _ in range(300):
        value, grads = weights.grad(objective)
        weights.step(grads, 0.5)
    b1_after = weights["b1"].value
    logits = forward(X)

print("final loss        :", round(float(value), 4))
print("b1 stayed all-zero:", bool(np.all(b1_after == 0.0)))
print("train accuracy    :", np.mean(np.argmax(logits, axis=1) == labels))

final loss        : 0.0008
b1 stayed all-zero: True
train accuracy    : 1.0


## More

The bundled demos (logistic regression, MLP, LayerNorm/Dropout, single-head Transformer block) train from scratch and are gradient-checked against finite differences:

```
python -m pycograd.examples
```